# Cleaning Downloaded Data from avian-influenza

Author: Alexander Maksiaev

Purpose: Clean downloaded data from Andersen Lab's avian-influenza GitHub, rename sequences according to convention

Notes:
* This file must be in the same folder as "utils.py"
* Before running this code, ensure that fork is updated

## Housekeeping

In [ ]:
input("Fork updated?")

In [ ]:
# Libraries

import os
import pandas as pd
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 


### Inputs and Paths

In [ ]:
# Dates
start_date = "11-01-2021"
end_date = "08-07-2026"
date_range = start_date + "--" + end_date

# Maintenance genotypes
# genotypes = ["A3"] # , "D1.1", "Not"]

# Make sure you have the correct paths

home = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu/"
downloads = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/"

references = home + "references/"
originals = downloads + "Andersen/"
metadata_folder = originals + "avian-influenza/metadata/"
temp_files = originals + "temp/"
ncbi_complete = downloads + "NCBI_Virus/complete/" + date_range + "_Antarctica_North_America_South_America/"

complete_files = originals + "complete/" + date_range + "/"
if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it
combined_files = downloads + "Combinations/NCBI_Virus_Andersen/" + date_range + "_Antarctica_North_America_South_America/"
if not os.path.exists(combined_files): # checking if the directory exists or not
    os.makedirs(combined_files) # if the directory is not present then create it


os.chdir(references)
state_ref = pd.read_csv("states_ref.csv")

# Get list of all genotypes
genotypes_df = pd.read_excel("genotype_key.xlsx")
genotypes = list(genotypes_df["Genotype"])
genotypes.append("Not")



## Read Metadata 

In [ ]:
# Get metadata from GitHub repo
os.chdir(metadata_folder)
metadata = pd.read_csv("SraRunTable_automated_normalized.tsv", delimiter="\t")
print(len(metadata)) 
print(metadata.columns)

# Find the name of the state sample was collected in
metadata["name_state"] = metadata["geo_loc_name"].apply(lambda x: x.split("/")[1]) # if len(x.split("/")[1]) > 0 else x.split("/")[0])

# Convert the dates to date format so we can compare
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["ReleaseDate"] >= dateutil.parser.parse(start_date).strftime("%Y-%m-%d")] # Find only >= last date using Release Date from metadata 
metadata = metadata[metadata["ReleaseDate"] <= dateutil.parser.parse(end_date).strftime("%Y-%m-%d")] # Find only <= update date using Release Date from metadata
metadata = metadata[metadata["is_retracted"] == False]

print(len(metadata)) 

## De-duplicate from NCBI Virus

In [ ]:
# Get NCBI Virus SRA sequences
ncbi_sras = []
for dirpath, dirs, files in os.walk(ncbi_complete):
    for file in files: # 8 * x genotypes
        file_name = os.path.join(dirpath, file)
        if ".fasta" in file_name:
            fasta_file = fasta_df_complete(file_name, state_ref) # Convert fasta file to dataframe
            sra_accessions = fasta_file["Identifier"]
            # Some accessions may not be SRA
            for value in sra_accessions.values:
                if "SRR" in value:
                    ncbi_sras.append(value)
            # Some duplicates may only be such because of duplicate isolates
            isolates = fasta_file["Isolate_Id"]
            for value in isolates.values:
                ncbi_sras.append(value)
            # Need partial isolates -- e.g. "012345-001" instead of "25-012345-001-original"
            partials = fasta_file["Partials"]
            for value in partials.values:
                ncbi_sras.append(value)
    break 

metadata["Partials"] = metadata["isolate"].apply(lambda x: partial_isolate(x) if x == x else x)

# Remove duplicates from Andersen
for value in ncbi_sras: # to remove
    if "SRR" in value:
        metadata = metadata[metadata["Run"] != value]
    
    metadata = metadata[metadata["isolate"] != value]
    metadata = metadata[metadata["Partials"] != value]
    # print(value)
    
print(len(metadata))

## Naming convention -- relabeling sequences


>[SRA_Accession]|A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype]|[geo_location]|[collection_date]|[host_type]|[genotype]

In metadata, we have: host, isolate, year, collection date, geo location, genotype

We need: host_type

host = Host

geo_loc_name = geo_loc_name

geo_location = country (abbreviated)-geo_loc_name (abbreviated) e.g. USA-MD

isolate = isolate

collection date = Collection_Date

serotype = H5N1, etc.

host type = animals_ref.csv (local)

genotype = genoflu_results.tsv > genotype

### Get genotype

In [ ]:
# Get genotype from genoflu_results.tsv

os.chdir(metadata_folder)

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")
genoflu_results = genoflu_results.rename(columns={"sample" : "Run"}) # Rename so we can merge

metadata = metadata.merge(genoflu_results, on="Run", how="inner") # Add genoflu results to dataframe, excluding runs without results
metadata_unassigned = metadata[metadata["Genotype"].str.contains("Not")]
metadata_genotypes = metadata[metadata["Genotype"].isin(genotypes)] # | metadata["Genotype"].str.contains("Not assigned")]

metadata = pd.concat([metadata_unassigned, metadata_genotypes]) if "Not" in genotypes else metadata_genotypes

print(len(metadata)) 
display(metadata)

In [ ]:


# Double-check state with genbank_mapping
os.chdir(metadata_folder)
genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")

# Merge with genbank_mapping
genbank_mapping = genbank_mapping.rename(columns={"sra_run":"Run"})
metadata = metadata.merge(genbank_mapping, how="left")



### Collection Dates

In [ ]:
# Get years from collection dates
metadata["years"] = metadata["Collection_Date"].apply(lambda x: dateutil.parser.parse(str(x), default=datetime(2000, 1, 1), fuzzy=True).year if x == x else str(x)) # Get year only from collection date

print(metadata["Collection_Date"])

### Get host type

In [ ]:
# create a mask, where is True if the host does not exist
mask = metadata["Host"].isna()

# choose between the original value and split isolate using the mask
metadata["Host"] = np.where(mask, 
                            metadata["isolate"].apply(lambda x: 
                                                      x if x != x or "/" not in x or len(x.split("/")) < 2 # If NaN or split isolate doesn't exist or split isolate is too short
                                                      else x.split("/")[1]), metadata["Host"]) # Provided that we have a long enough isolate with "/" in them, get the host

metadata["Host"] = metadata["Host"].apply(lambda x: x.lower() if x == x else x) # make sure all characters are lowercase

# Create animals ref if needed
unique_animals_all = sort_animals_andersen(metadata)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

# print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2
common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and animal == animal: # If animal exists in dataframe and isn't NaN
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print("New animals to add to reference:", different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# Unlikely for different_animals to be longer than the dataframe, but just in case
else:
    number_of_times_to_add_nan = len(different_animals) - len(animals_df)
    for i in range(number_of_times_to_add_nan):
        empty_rows = pd.DataFrame(np.nan, index=range(number_of_times_to_add_nan), columns=animals_df.columns)
        animals_df = pd.concat([animals_df, empty_rows], ignore_index=True)

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 

print(metadata["Host"])
# print(metadata["isolate"])

In [ ]:
# Ensure that user checks animal output
input("Check animals output. Afterwards, press ESCAPE to continue.")

In [ ]:
# Get animals from animal reference
os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata, animals_ref) # Get host type

### Get specific geolocation

In [ ]:
# Format: USA-[state abbreviation], e.g. USA-MD
def geo_location_get(metadata: pd.DataFrame, state_ref_file: str = "states_ref.csv") -> pd.DataFrame:

    state_ref = pd.read_csv(state_ref_file)

    # Normalize the locations

    # metadata["name_state_genbank"] = metadata["genbank_name"].apply(lambda x: x.split("/")[2] if x == x else x)

    metadata["Geo_Location_Normalized"] = metadata["Geo_Location"].apply(geo_location_normalize)

    metadata["Geo_Location_Country"] = metadata["Geo_Location_Normalized"].apply(lambda x: x.split("-")[0])

    # Get country
    metadata["Geo_Location_Country"] = metadata["Geo_Location_Country"].apply(lambda x: state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Country'].iloc[0]
                                                                              if len(state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Country']) > 0
                                                                              else state_ref.loc[state_ref["State"].str.contains(x), 'Country'].iloc[0]
                                                                              if len(state_ref.loc[state_ref["State"].str.contains(x), 'Country']) > 0
                                                                              else x)

    # Get state
    metadata["Geo_Location_State_med"] = metadata["Geo_Location_Normalized"].apply(lambda x: x.split("-")[-1])

    # try:    
    metadata["Geo_Location_State_USA"] = metadata["Geo_Location_State_med"].apply(lambda x: 
        # # If "x" is the abbreviated state (e.g. "MD")
        state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Abbreviation'].iloc[0]
        if len(state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Abbreviation']) > 0 and len(x) > 0
        # # If "x" is the state name (e.g. "Maryland")
        else state_ref.loc[state_ref["State"].str.contains(x), 'Abbreviation'].iloc[0]
        if len(state_ref.loc[state_ref["State"].str.contains(x), 'Abbreviation']) > 0 and len(x) > 0
        # # If "x" has neither the state abbreviation nor the full state name
        else x)
    # except:
    #     print("Could not find states in reference.")

    # If there is no state in the metadata, try the strain name
    # try:
    # For each Geo_Location_State that has len(value) == 0 (i.e., nan or "")
    metadata["Geo_Location_State"] = metadata["Run"].apply(lambda x: # These should be unique
        # Find the corresponding strain name
        metadata[metadata["Run"] == x]["genbank_name"].values[0].split["/"][1] # We will know whether human or non-human -- check Host_Type
        if len(metadata[metadata["Run"] == x]["Geo_Location_State_USA"].values[0]) == 0 # state 
        and metadata[metadata["Run"] == x]["genbank_name"].values[0] == metadata[metadata["Run"] == x]["genbank_name"].values[0]
        and metadata[metadata["Run"] == x]["Host_Type"].values[0] == "human" # If human, do strain name [1] 
        else 
        # Find state
        metadata[metadata["Run"] == x]["genbank_name"].values[0].split["/"][2]
        if len(metadata[metadata["Run"] == x]["Geo_Location_State_USA"].values[0]) == 0 # state 
        and metadata[metadata["Run"] == x]["genbank_name"].values[0] == metadata[metadata["Run"] == x]["genbank_name"].values[0]
        and metadata[metadata["Run"] == x]["Host_Type"].values[0] != "human" # If nonhuman, do strain name [2]
        # Otherwise, use Geo_Location_State_USA
        else metadata[metadata["Run"] == x]["Geo_Location_State_USA"].values[0])
    # except:
    #     print("Could not find states in strain names.")
    
    

    metadata["Geo_Location_New"] = metadata["Geo_Location_Country"] + "-" + metadata["Geo_Location_State"] 

    # If USA-, delete -
    metadata["Geo_Location_New"] = metadata["Geo_Location_New"].apply(lambda x: x.split("-")[0] 
    if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] 
    else x)

    # Log metadata
    return metadata


In [ ]:
os.chdir(references)
# Get the name of the state, unless it's not in genbank_mapping -- then get it from normalized metadata
metadata["name_state_genbank"] = metadata["genbank_name"].apply(lambda x: x.split("/")[2] if x == x and len(x.split("/")) > 2
                                                                else x)
metadata["name_state_genbank"] = metadata["name_state_genbank"].fillna(metadata["name_state"])

metadata["Geo_Location"] = metadata["name_state_genbank"]
metadata = geo_location_get(metadata=metadata)

display(metadata)

### Make names using all the attributes we collected

In [ ]:
# Make names

# metadata["isolate_name"] = metadata["genbank_name"]

metadata = metadata.fillna("") # Make sure the entire name does not become "NaN"

metadata["isolate_name"] = np.where(metadata["genbank_name"] == "", "A/" + metadata["Host"].apply(lambda x: x.replace(" ", "_")) + "/" + metadata["geo_loc_name"].apply(lambda x: x.split("/")[1] if len(x.split("/")[1]) > 0 else x.split("/")[0]) + "/" + metadata["Sample Name"] + "/" + metadata["years"].apply(lambda x: str(x)), metadata["genbank_name"])


names = ">" + metadata["Run"] + "|" + metadata["isolate_name"] + "|" + metadata["serotype"] + "|" + metadata["Geo_Location_New"] + "|" + metadata["Collection_Date"].apply(lambda x: str(x) if "-" not in str(x) else str(dateutil.parser.parse(str(x), default=datetime(2000, 1, 1)).strftime("%Y")) if dateutil.parser.parse(str(x), default=datetime(2000, 1, 1)).month == datetime(2000, 1, 1).month and dateutil.parser.parse(str(x), default=datetime(2000, 1, 1)).day == datetime(2000, 1, 1).day else dateutil.parser.parse(str(x), default=datetime(2000, 1, 1)).strftime("%Y-%m-%d")) + "|" + metadata["Host_Type"] + "|" + metadata["Genotype"]

metadata["Name"] = names

display(metadata["Name"])

In [ ]:
# Drop duplicate runs 

metadata["Partials"] = metadata["isolate"].apply(partial_isolate)
metadata = metadata.drop_duplicates(subset=["Partials", "years"], keep="last") # Isolates may be identical, first=NCBI Virus, last=Andersen

In [ ]:
os.chdir(complete_files)

print(metadata)
# Save metadata
metadata.to_csv("Andersen_metadata_" + date_range + ".csv")

## Make FASTA files

In [ ]:
# Get information to create the fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

# Create pairs of genotypes and segments, e.g. B3.13_HA
for genotype in genotypes: # ["B3.13", "D1.1"]:
    for segment in segments:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata[metadata["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata[metadata["Run"] == run].loc[:, "Genotype"].values[0].split(" ")[0] # Make sure "Not assigned" stays as "Not"

                    if "Not" in genotype:
                        genotype = "Unassigned"
                    # print(header)
                    print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()
        break 

## Concatenate with NCBI Virus

In [ ]:
# Create fasta files 
 
os.chdir(complete_files)
names = []
for pair in fasta_files.keys():
    if len(fasta_files[pair]) > 0: # If this isn't empty

        output_path = complete_files + pair + "_" + date_range + ".fasta"

        output_file = open(output_path, "w")
        for item in fasta_files[pair]:
            try:
                name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
            except:
                name = str(item[0])
            print(name)
            name = name.replace(" ", "_")
            names.append(name)
            # First is header, second is sequence
            output_file.write(name + "\n")
            output_file.write(item[1])
        output_file.close()

print(len(names)/8)

In [ ]:
# Concatenate with new NCBI Virus sequences

os.chdir(combined_files)
segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]

andersen = home + "Andersen/complete/" + date_range + "/"

# NCBI Virus files
filenames_ncbi = []

for dirpath, dirs, files in os.walk(ncbi_complete): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file) # Get file name
        filenames_ncbi.append(file_name)
    break 

# Andersen files
filenames_andersen = []
for dirpath, dirs, files in os.walk(complete_files): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file) # Get file name
        filenames_andersen.append(file_name)
    break 

print(filenames_andersen)

common_genotypes = set()
# Concatenate the two -- should not have any overlap due to dates and deduplication 
for a_file in filenames_andersen:
    a_file_name = a_file.split("/")[-1].split("_")[0] + "_" + a_file.split("/")[-1].split("_")[1]
    print(a_file_name)
    for nv_file in filenames_ncbi:
        nv_file_name = nv_file.split("/")[-1].split("_")[0] + "_" + nv_file.split("/")[-1].split("_")[1]
        print(nv_file_name)
        if a_file_name == nv_file_name: # We have a common genotype
            common_genotypes.add(a_file_name)
            filenames = [a_file, nv_file]
            with open(combined_files + a_file_name + "_" + date_range + ".fasta", 'w') as outfile:
                for fname in filenames:
                    with open(fname) as infile:
                        for line in infile:
                            outfile.write(line)
            infile.close()
            outfile.close()

# print(common_genotypes)

for a_file in filenames_andersen:
    partial_filename_a = a_file.split("/")[-1].split("_")[0] + "_" + a_file.split("/")[-1].split("_")[1]
    
    # If genotype not found in one of the datasets, include it as well
    if partial_filename_a not in common_genotypes and ".fasta" in a_file:
        print(partial_filename_a)
        for segment in segments:
            with open(combined_files + partial_filename_a + "_" + date_range + ".fasta", 'w') as outfile2:
                with open(a_file) as infile2:
                    for line in infile2:
                        outfile2.write(line)
                infile2.close()
                outfile2.close()

for a_file in filenames_ncbi:
    partial_filename_a = a_file.split("/")[-1].split("_")[0] + "_" + a_file.split("/")[-1].split("_")[1]
    
    # If genotype not found in one of the datasets, include it as well
    if partial_filename_a not in common_genotypes and ".fasta" in a_file:
        for segment in segments:
            print(partial_filename_a)
            with open(combined_files + partial_filename_a + "_" + date_range + ".fasta", 'w') as outfile3:
                with open(a_file) as infile3:
                    for line in infile3:
                        outfile3.write(line)
                infile3.close()
                outfile3.close()